# Chayanta: cuento en quechua boliviano — dos interpretaciones y efectos

**Pulsa ▶ en la única celda.** Generará un MP3 completo con dos perfiles vocales (Romaldina y Casiana), diálogos de los sapitos, croares sintéticos, lluvia y un ZIP descargable. La narración es más pausada que en la versión anterior.

**Precisión:** `facebook/mms-tts-quh` contiene **una sola voz de origen**. Las dos interpretaciones se crean ajustando tono y velocidad, no son grabaciones de dos mujeres. Puede requerir revisión de pronunciación por un quechuahablante de Chayanta. Los apóstrofes se conservan en el texto.

**Licencia del modelo:** CC BY-NC 4.0, solo uso no comercial. [Modelo oficial](https://huggingface.co/facebook/mms-tts-quh).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║ UNA SOLA CELDA: 2 timbres distintos → sapitos y lluvia → MP3 + WAV ║
# ╚══════════════════════════════════════════════════════════════════╝
# Modelo: facebook/mms-tts-quh (quechua sudboliviano; CC BY-NC 4.0).
# Se requiere internet para instalar dependencias y descargar el modelo.
import subprocess, sys, os, shutil, re, json, zipfile, random, math
from pathlib import Path

print("① Instalando bibliotecas de voz (sin instalar nada en tu PC)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "transformers>=4.46,<5", "imageio-ffmpeg>=0.5", "safetensors"])

import numpy as np
import torch
from scipy.io import wavfile
from scipy import signal
from transformers import VitsModel, AutoTokenizer, set_seed
from IPython.display import display, Audio
from imageio_ffmpeg import get_ffmpeg_exe

MODEL_ID = "facebook/mms-tts-quh"
DEST = Path("/content/CHAYANTA_QUECHUA_DOS_VOCES") if Path("/content").exists() else Path("CHAYANTA_QUECHUA_AUDIO")
DEST.mkdir(parents=True, exist_ok=True)
OUTPUT_ZIP = DEST.parent / "CHAYANTA_SAPITOS_DOS_VOCES_PRUEBA.zip"
SEED = 20261108
set_seed(SEED)
random.seed(SEED)
rng = np.random.default_rng(SEED)

# Texto íntegro y original. Los apóstrofes se conservan en el archivo de texto.
ESCENAS = [('narradora', 'Unay pachapi, Chayanta llaqtapi, jatun urqukunap chawpinpi, Casiana sutiyuq uj warmi wawa tiyakuq karqa. Payqa awichan Romaldinawan kuska kawsakuq karqa.'), ('narradora', 'Chay llaqtapi runakuna papa, hawas, awina, siwarata tarpukuq kanku. Ichaqa chay watapi para mana chayamurqanchu.'), ('narradora', "Jallp'aqa ch'akirirqan. Chakrakunapi tarpuykuna wañuriyta qallariq kanku. Chayanta mayupis asllataña yaku puriq."), ('narradora', "Uj ch'isi, Casianaqa awichanwan mayu pataman rirqa."), ('casiana', 'Awichay Romaldina, imaraykutaq mana paramunchu?'), ('narradora', 'nispa tapurqa.'), ('narradora', "Awichanqa jallp'ata qhawarispa, llamp'u simiwan kutichirqa:"), ('romaldina', "Wawitay Casiana, ñawpaq awichanchikkuna uj sumaq willakuyta saqiwanchik. Jamp'atuchakunaqa, jallp'a ch'akiptin, parata mañakuspa takinku, niq kanku."), ('casiana', "Jamp'atuchakunachu parata mañakunku, awichay?"), ('romaldina', 'Arí, wawitay. Uchuylla kaspapas, jatun sunquyuq kanku. Paykunaqa yakuta munanku, chakrakuna kawsananta munanku.'), ('narradora', "Chay ratupi, mayu patamanta uj uchuy jamp'atu rikhurimurqa."), ('sapito_1', 'Kruq... kruq... kruq!'), ('narradora', "Casianaqa ch'inyaspa uyarirqan."), ('narradora', "Jamp'atuchaqa mana saykuspa takirqan. Chaymanta uj jamp'atu kutichimurqa, chaymantataq juk jamp'atu."), ('narradora', "Mayu patapi, rumikunap ukhunpi, chakrakunap qayllanpi, tukuy jamp'atuchakuna takiyta qallarirqanku."), ('narradora', "Chay tutapi, Casianaqa jamp'atuchakunaq rimayninta uyariyta atirqa."), ('narradora', "Ñawpaq jamp'atuqa sinch'ita waqaspa nirqa:"), ('sapito_1', "Kruq, kruq! Para mamay, uyariwayku! Jallp'anchikqa ch'akisqa! Yaku mana kanchu!"), ('narradora', "Juk jamp'atuchataq kutichirqa:"), ('sapito_2', 'Kruq, kruq! Papaqa yakuta munan! Hawasqa yakuta munan! Awina, siwarapis yakuta munanku!'), ('narradora', "Tukuy jamp'atuchakuna kuska qaparirqanku:"), ('sapitos_coro', 'Paramuchun! Paramuchun! Chayanta mayuman yaku kutimuchun! Pachamamanchik kusisqa kachun!'), ('narradora', "Casianaqa awichanpa makinta jap'irqan."), ('casiana', "Awichay, uyarinkichu? Jamp'atuchakunaqa llakikushanku. Jallp'anchikpaq parata mañakushanku."), ('narradora', 'Romaldinaqa ñawinkunapi wiqiwan kutichirqa:'), ('romaldina', "Arí, wawitay. Kay jamp'atuchakunaqa yachachiwanchik: tukuy kawsayqa yakuta munan. Runapis, uywapas, sach'apas, uchuy jamp'atuchapas."), ('narradora', 'Chaymanta wayraqa phukuyta qallariqa.'), ('narradora', 'Janaq pachapi phuyukuna tantakurqanku. Killaqa phuyukunap qhipanpi pakakurqa.'), ('narradora', 'Uj chhillchi para urmayta qallariqa.'), ('narradora', "Casianaq uyampiman uj yaku sut'u urmarqa. Chaymanta juk, chaymanta astawan."), ('casiana', 'Awichay! Paramushan! Paramushan!'), ('narradora', 'Romaldinaqa makinkunata kicharispa janaq pachata qhawarqa.'), ('romaldina', 'Añay, Pachamama! Añay, para mamay!'), ('narradora', "Para mayuta junt'achirqan. Ch'akirisqa jallp'aqa yakuta ukyarqa. Papa chakrakuna, hawas, awina, siwaraqa musuq kawsayta tarirqanku."), ('narradora', "Jamp'atuchakunaqa kusisqa takillasharqanku:"), ('sapitos_coro', 'Kruq, kruq! Yaku chayamunña! Kawsay kutimunña!'), ('narradora', "Paqarin punchawpi, inti lloqsimuptin, Chayanta llaqtapi tukuy ima sumaqta k'anchariq karqa."), ('narradora', "Casianaqa awichanta marq'arqa."), ('casiana', "Awichay, jamp'atuchakunaqa parata apamurqankuchu?"), ('narradora', 'Romaldinaqa asirispa nirqa:'), ('romaldina', 'Ichapis, wawitay. Ichapis para chayamunanta ñawpaqta yacharqanku. Ichaqa uj imata mana qunqananchikchu: uchuylla kawsaypas jatun yuyayta yachachiwanchik.'), ('narradora', "Chay punchawmanta pacha, Casianaqa sapa para qallariypi jamp'atuchakunaq takiyninta uyarikuq karqa."), ('narradora', "Payqa yacharqa: Chayanta llaqtapi jamp'atuchakunaq takiyninqa mana ch'ulla takiyllachu karqa."), ('narradora', "Chayqa jallp'aq suyakuynin karqa. Chayqa kawsaypaq mañakuynin karqa. Chayqa parata suyaq tukuy sunqukunap takiynin karqa.")]

TEXTO_ORIGINAL = "CHAYANTA LLAQTAPI JAMP'ATUCHAKUNAQ TAKIYNIN\n\n" + "\n\n".join(
    f"[{personaje.upper()}]\n{texto}" for personaje, texto in ESCENAS
) + "\n"
(DEST / "CUENTO_ORIGINAL_QUECHUA.txt").write_text(TEXTO_ORIGINAL, encoding="utf-8")
(DEST / "ESCENAS_QUECHUA.json").write_text(
    json.dumps([{"voz": p, "texto": t} for p,t in ESCENAS], ensure_ascii=False, indent=2), encoding="utf-8")

print("② Cargando modelo auténtico de quechua sudboliviano (quh)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = VitsModel.from_pretrained(MODEL_ID)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
rate = int(model.config.sampling_rate)
print(f"   Modelo: {MODEL_ID} | Dispositivo: {device} | Muestra: {rate} Hz")

# El motor MMS normaliza internamente la puntuación (incluidos apóstrofes).
# El texto escrito queda intacto, pero la pronunciación exacta debe revisarla
# un hablante nativo de quechua de Chayanta.

# Separa frases muy largas para evitar generar audio demasiado largo de golpe.
def dividir_texto(texto, max_caracteres=165):
    frases = re.split(r"(?<=[.!?])\s+", texto.strip())
    partes = []
    fragmento = ""
    for frase in frases:
        if len(frase) > max_caracteres:
            # Divide sin cortar palabras ni separar letras con apóstrofes.
            words = frase.split()
            for word in words:
                intento = (fragmento + " " + word).strip()
                if len(intento) > max_caracteres and fragmento:
                    partes.append(fragmento)
                    fragmento = word
                else:
                    fragmento = intento
        elif len((fragmento + " " + frase).strip()) > max_caracteres and fragmento:
            partes.append(fragmento)
            fragmento = frase
        else:
            fragmento = (fragmento + " " + frase).strip()
    if fragmento:
        partes.append(fragmento)
    return partes


def sintetizar(texto, personaje):
    # El modelo VITS modifica la velocidad desde la síntesis, no acelerando el audio final.
    model.speaking_rate = 0.75 if personaje in ('narradora','romaldina') else 0.87
    inp = tokenizer(texto, return_tensors="pt")
    inp = {k: v.to(device) for k,v in inp.items()}
    if inp["input_ids"].numel() == 0:
        raise RuntimeError("El modelo no reconoció el texto: " + texto[:60])
    with torch.inference_mode():
        onda = model(**inp).waveform.squeeze().float().cpu().numpy()
    if onda.size == 0 or not np.isfinite(onda).all():
        raise RuntimeError("Audio inválido al sintetizar: " + texto[:60])
    return onda.astype(np.float32)


def silencio(s):
    return np.zeros(int(s * rate), dtype=np.float32)


def croar():
    # Efecto original/sintético de sapito. No es grabación de un animal real.
    duration = 0.43
    t = np.arange(int(duration * rate)) / rate
    f0 = 185 - 80*(t/duration) + 18*np.sin(2*np.pi*9*t)
    phase = 2*np.pi*np.cumsum(f0)/rate
    env = np.sin(np.pi * np.clip(t/duration, 0, 1))**2
    sound = (0.12*np.sin(phase) + 0.04*np.sin(2*phase))*env
    return sound.astype(np.float32)


def cambiar_timbre(audio, personaje, ix):
    # El modelo tiene UNA voz. Diferenciación MODERADA mediante tono:
    # son variantes procesadas, no locutores humanos independientes.
    factores = {"narradora":1.20, "romaldina":1.20, "casiana":1.48,
                "sapito_1":0.90, "sapito_2":0.95, "sapitos_coro":0.92}
    fac = factores[personaje]
    if abs(fac-1)<1e-6:
        return audio
    temp_in = DEST/f"_temporal_{ix}.wav"
    temp_out = DEST/f"_temporal_{ix}_edit.wav"
    wavfile.write(str(temp_in), rate, audio)
    # FFmpeg: cambiar pitch conservando aproximadamente la duración.
    af = f"asetrate={int(rate*fac)},aresample={rate},atempo={1/fac:.6f}"
    try:
        subprocess.run([get_ffmpeg_exe(),"-hide_banner","-loglevel","error","-y",
                        "-i",str(temp_in),"-af",af,str(temp_out)],check=True)
        out_rate, out_audio = wavfile.read(str(temp_out))
        if out_rate != rate:
            raise RuntimeError("Tasa de muestreo inesperada")
        return out_audio.astype(np.float32)
    finally:
        temp_in.unlink(missing_ok=True)
        temp_out.unlink(missing_ok=True)


print(f"③ Generando narración y diálogos ({len(ESCENAS)} escenas)...")
segmentos = []
muestras = {}
for ix, (voz, texto) in enumerate(ESCENAS, start=1):
    trozos = [sintetizar(p, voz) for p in dividir_texto(texto)]
    voz_audio = np.concatenate([x for chunk in trozos for x in (chunk, silencio(0.19))])
    voz_audio = cambiar_timbre(voz_audio, voz, ix)
    if voz in ("romaldina", "casiana") and voz not in muestras:
        muestras[voz] = voz_audio.copy()
    if voz.startswith("sapito"):
        efecto = np.concatenate([croar(), silencio(0.12)])
        voz_audio = np.concatenate([efecto, voz_audio])
    if voz == "sapitos_coro":
        voz_audio = np.concatenate([croar(), silencio(0.07), voz_audio])
    # Evitar saturación y mantener un nivel moderado.
    peak = max(1.0, float(np.max(np.abs(voz_audio))))
    voz_audio = (voz_audio / peak * 0.86).astype(np.float32)
    segmentos.extend([voz_audio, silencio(0.48 if voz in ("narradora", "romaldina") else 0.56)])
    if ix % 5 == 0 or ix == len(ESCENAS):
        print(f"   Voz generada: {ix}/{len(ESCENAS)}")

narracion = np.concatenate(segmentos).astype(np.float32)

# Ambiente discreto de lluvia en la escena final; efecto sintético propio.
primera_escena_de_lluvia = 29
comienzo_lluvia = sum(len(x) for x in segmentos[:2*(primera_escena_de_lluvia-1)])
if comienzo_lluvia < len(narracion):
    n = len(narracion) - comienzo_lluvia
    lluvia = rng.normal(0, 1, n).astype(np.float32)
    lluvia = signal.lfilter([0.04], [1, -0.98], lluvia).astype(np.float32)
    lluvia = lluvia / max(1e-5, float(np.max(np.abs(lluvia))))
    entrada = np.minimum(1.0, np.arange(n, dtype=np.float32) / max(1, rate*4))
    salida = np.minimum(1.0, np.arange(n, dtype=np.float32)[::-1] / max(1, rate*5))
    narracion[comienzo_lluvia:] += lluvia * entrada * salida * 0.035
narracion = (narracion / max(1.0, float(np.max(np.abs(narracion)))) * 0.97).astype(np.float32)

print("④ Guardando WAV y MP3...")
wav_path = DEST/"CHAYANTA_SAPITOS_QUECHUA_DOS_VOCES_PRUEBA.wav"
mp3_path = DEST/"CHAYANTA_SAPITOS_QUECHUA_DOS_VOCES_PRUEBA.mp3"
wavfile.write(str(wav_path), rate, narracion)
subprocess.run([get_ffmpeg_exe(),"-hide_banner","-loglevel","error","-y",
                "-i",str(wav_path),"-codec:a","libmp3lame","-b:a","192k",
                str(mp3_path)],check=True)
if not wav_path.is_file() or not mp3_path.is_file() or mp3_path.stat().st_size<10000:
    raise RuntimeError("Fallo de exportación de audio")

mins = len(narracion)/rate/60
licencia = """Narración de Chayanta (quechua boliviano)
Modelo de voz: facebook/mms-tts-quh, Meta MMS-TTS, licencia CC BY-NC 4.0.
Uso del modelo: NO COMERCIAL; atribuir a sus creadores y cumplir la licencia.
La historia es original y se incluye completa en CUENTO_ORIGINAL_QUECHUA.txt.
Voces: dos perfiles procesados de la misma voz TTS, no son locutoras independientes.\nRomaldina y narradora: tono adulto y ritmo lento. Casiana: tono infantil y velocidad suave.
Croares y lluvia: efectos sintetizados en este cuaderno; no grabaciones reales.
Pronunciación: revisar con hablante nativo de quechua local antes de publicar.
La puntuación y los apóstrofes se preservan en el texto, pero el tokenizador
MMS puede eliminarlos durante la normalización para sintetizar audio.
"""
(DEST/"LEEME_LICENCIA_Y_LIMITACIONES.txt").write_text(licencia,encoding="utf-8")

# Muestras individuales para comprobar las dos interpretaciones vocales.
for personaje, nombre in (("romaldina", "PRUEBA_ABUELITA_ROMALDINA.mp3"),
                          ("casiana", "PRUEBA_NIÑA_CASIANA.mp3")):
    p = DEST/nombre
    p_wav = DEST/(p.stem + ".wav")
    wavfile.write(str(p_wav), rate, muestras[personaje])
    subprocess.run([get_ffmpeg_exe(), "-hide_banner", "-loglevel", "error", "-y",
                    "-i", str(p_wav), "-codec:a", "libmp3lame", "-b:a", "192k", str(p)], check=True)
    p_wav.unlink(missing_ok=True)

if OUTPUT_ZIP.exists(): OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for name in ("CHAYANTA_SAPITOS_QUECHUA_DOS_VOCES_PRUEBA.mp3",
                 "CHAYANTA_SAPITOS_QUECHUA_DOS_VOCES_PRUEBA.wav",
                 "PRUEBA_ABUELITA_ROMALDINA.mp3", "PRUEBA_NIÑA_CASIANA.mp3",
                 "CUENTO_ORIGINAL_QUECHUA.txt", "ESCENAS_QUECHUA.json",
                 "LEEME_LICENCIA_Y_LIMITACIONES.txt"):
        zf.write(DEST/name,arcname=name)
with zipfile.ZipFile(OUTPUT_ZIP) as zf:
    if zf.testzip() is not None:
        raise RuntimeError("Fallo de integridad del ZIP")
print(f"✅ Audio generado y comprobado | {mins:.2f} minutos")
print(f"✅ MP3: {mp3_path.name} ({mp3_path.stat().st_size/1e6:.1f} MB)")
print(f"✅ WAV: {wav_path.name} ({wav_path.stat().st_size/1e6:.1f} MB)")
print(f"✅ Paquete: {OUTPUT_ZIP.name}")
print("Escucha el MP3 completo aquí:")
display(Audio(narracion, rate=rate))

# Descarga directa automática para el navegador, sin pasos extra.
try:
    from google.colab import files
    print("⑤ Iniciando descarga automática del ZIP con todo el audio...")
    files.download(str(OUTPUT_ZIP))
except ImportError:
    print("No estás en Colab. El ZIP quedó guardado en:", OUTPUT_ZIP.resolve())
